## Hierarchically consistent forecasting with package sizes

1. We forecast on multiple hierarchical levels with a dataset containing demands.
2. After forecasting the time series individually on the three different hierarchical levels, we will reconcile the forecasts to achieve hierarchically consistent forecasts while keeping detected package sizes.


For more information about hierarchically consistent forecasts have a look at this [notebook](make_forecasts_consistent.ipynb).

In [1]:
import time

import pandas as pd

import futureexpert.checkin as checkin
from futureexpert import (DataDefinition,
                          ExpertClient,
                          FileSpecification,
                          ForecastingConfig,
                          MethodSelectionConfig,
                          PreprocessingConfig,
                          ReconciliationConfig,
                          ReconciliationMethod,
                          ReportConfig,
                          TsCreationConfig)
from futureexpert.forecast_consistency import (ReconciliationConfig,
                                               ReconciliationMethod)

client = ExpertClient()

INFO:futureexpert.expert_client:Successfully logged in for group group-expert.


## CHECK-IN configuration
In this step, we transform our CSV data into structured time series. First, we define the data structure. For each column, we specify whether it contains the date, grouping information, or value data. 
During the time series creation process, we choose to aggregate the data at the CUSTOMER, MATERIAL and REGION level. We also enable the `save_hierarchy` so the information about the hierarchical structure persists. 

For more information about the CHECK-IN Process see the [checkin configuration options](./checkin_configuration_options.ipynb).

In [2]:
df = pd.read_csv('../use_cases/demand_planning/demo_demand_planning_data.csv',sep=';')
df.DATE = pd.to_datetime(df.DATE)
global_max_date = df.DATE.max()

# Filter groups where the group's max date equals the global max date
df_filtered = df.groupby(['CUSTOMER', 'MATERIAL', 'REGION']).filter(
    lambda x: x.DATE.max() == global_max_date
)

df_filtered.DATE = df_filtered['DATE'].dt.strftime('%d.%m.%Y')

In [3]:
actuals_version_id = client.check_in_time_series(raw_data_source=df_filtered,
                                                 file_specification=FileSpecification(delimiter=';', decimal='.'),
                                                  data_definition=DataDefinition(date_column=checkin.DateColumn(name='DATE', format='%d.%m.%Y', name_new = 'Date'),
                                                                                 value_columns=[checkin.ValueColumn(name='DEMAND', name_new='Demand')],
                                                                                 group_columns=[checkin.GroupColumn(name='CUSTOMER', name_new='Customer'),
                                                                                                checkin.GroupColumn(name='MATERIAL', name_new='Material'),
                                                                                                checkin.GroupColumn(name='REGION', name_new='Region')]),
                                                  config_ts_creation=TsCreationConfig(time_granularity='monthly',
                                                                                      start_date='2007-10-01',
                                                                                      end_date='2024-06-01',
                                                                                      value_columns_to_save=['Demand'],
                                                                                      grouping_level=['Region', 'Material'],
                                                                                      missing_value_handler='setToZero',
                                                                                      save_hierarchy=True,
                                                                                      filter=[checkin.FilterSettings(type='exclusion', variable='Material', items=['C011414576'])]))


INFO:futureexpert.expert_client:Transforming input data...
INFO:futureexpert.expert_client:Creating time series using CHECK-IN...
INFO:futureexpert.expert_client:Finished time series creation.


## Forecasting with hierarchical consistency and package sizes

With CHECK-IN having prepared our time series, we are ready to configure the FORECAST settings:
- We define a set of forecasting methods with fast performances for the upper two hierarchical levels.
- For the deepest hierarchical level we only use a foundation model.
- We activate detect_quantization to detect the package sizes.

In [4]:
fc_methods = ['LinearRegression', 'Naive', 'MA(granularity)', 'MA(3)', 'MA(season lag)', 'FoundationModel']
fc_methods_per_level = { 3: ['FoundationModel']}
fc_report_config = ReportConfig(title='Monthly Demand Forecast on Multiple Hierarchical Levels and Package Size',
                                preprocessing=PreprocessingConfig(detect_outliers=True,
                                                                    replace_outliers=True,
                                                                    detect_changepoints=True,
                                                                    detect_quantization=True),
                                forecasting=ForecastingConfig(fc_horizon=12,
                                                                lower_bound=0,
                                                                use_ensemble=False,
                                                                confidence_level=0.90),
                                method_selection=MethodSelectionConfig(number_iterations=3,
                                                                        refit=True,
                                                                        step_weights={1: 1., 2: 1., 3: 1.,
                                                                                        4: 0.5, 5: 0.5, 6: 0.5},
                                                                        forecasting_methods=fc_methods,
                                                                        forecasting_methods_per_hierarchy_level=fc_methods_per_level),
                                max_ts_len=72)


The process of making time series hierarchically consistent is called hierarchical reconciliation. We need to specify the specific reconciliation method and activate the package size constraint.

In [5]:

hcfc_config = ReconciliationConfig(
    method=ReconciliationMethod.TOP_DOWN_AVERAGE_FORECAST_PROPORTION,
    fallback_methods=[],
    round_forecast_to_package_size=True
)

In [6]:
forecast_identifier = client.start_forecast(
    version=actuals_version_id, config=fc_report_config, reconciliation_config=hcfc_config)
print(forecast_identifier)

INFO:futureexpert.expert_client:Preparing data for forecast...
INFO:futureexpert.expert_client:Finished data preparation for forecast.
INFO:futureexpert.expert_client:Started creating forecasting report with FORECAST...
INFO:futureexpert.expert_client:Report created with ID 151849. Forecasts are running...
INFO:futureexpert.expert_client:Preparing data for forecast consistency...
INFO:futureexpert.expert_client:Finished data preparation for forecast consistency.
INFO:futureexpert.expert_client:Started creating hierarchical reconciliation for consistent forecasts...
INFO:futureexpert.expert_client:Report created with ID 151850. Reconciliation is running...


report_id=151850 settings_id=151892 prerequisites=[ReportIdentifier(report_id=151849, settings_id=151891)]


In [7]:
# Watch the current status of the forecasting report
while not (current_status := client.get_report_status(id=forecast_identifier)).is_finished:
    time.sleep(20)  # Wait between status requests
# Retrieve the results of the forecasting report
current_status.print()
assert current_status.results.error == 0, \
    'At least one forecast failed. All forecasts must succeed to complete this notebook.'

Status of report "Monthly Demand Forecast on Multiple Hierarchical Levels and Package Size" of type "forecast":
 100 % are finished 
 19 time series requested for calculation 
 19 time series finished 
 2 time series without evaluation 
 0 time series ran into an error

Error reasons:
  [NoEvaluation] 
    Affected time series (2): Demand-Europe-C031595988, Demand-North America-C031595988
Status of report "Monthly Demand Forecast on Multiple Hierarchical Levels and Package Size" of type "hierarchical-forecast":
 100 % are finished 
 1 runs requested for calculation 
 1 runs finished 
 0 runs without evaluation 
 0 runs ran into an error


### Forecast results

In [8]:
results = client.get_fc_results(id=forecast_identifier, include_backtesting=True, include_k_best_models=10)
results.forecast_results.sort(key=lambda result: result.input.actuals.name)
[i.input.actuals.name for i in results.forecast_results]

['Demand',
 'Demand-Europe',
 'Demand-Europe-C011414575',
 'Demand-Europe-C024897037',
 'Demand-Europe-C031595988',
 'Demand-Europe-C042056164',
 'Demand-Europe-C046568618',
 'Demand-Europe-C048529686',
 'Demand-North America',
 'Demand-North America-C010887784',
 'Demand-North America-C011414575',
 'Demand-North America-C017968454',
 'Demand-North America-C024897037',
 'Demand-North America-C026568362',
 'Demand-North America-C031595988',
 'Demand-North America-C042056164',
 'Demand-North America-C046568618',
 'Demand-North America-C048097345',
 'Demand-North America-C048529686']

## Check that detected package sizes are used in hierarchical consistent forecast

In [9]:
def assert_forecasts_match_quantization(results):
      """
      Verify that all forecast values are multiples of the detected quantization.
      
      Parameters
      ----------
      results
          The forecast results to validate
      """
      for forecast_result in results.forecast_results:
          quantization = forecast_result.ts_characteristics.quantization

          # Only check if quantization was detected
          if quantization is not None:
              ts_name = forecast_result.input.actuals.name
              best_model = forecast_result.best_model

              if best_model is None:
                  print(f"{ts_name}: No best model found")
                  continue

              # Check each forecast value
              for forecast in best_model.forecasts:
                  forecast_value = forecast.point_forecast_value

                  # Check if the value is a multiple of quantization
                  # Use modulo with small tolerance for floating point comparison
                  remainder = forecast_value % quantization

                  if remainder > 1e-6 and (quantization - remainder) > 1e-6:
                      raise AssertionError(
                          f"{ts_name}: Forecast value {forecast_value} at "
                          f"{forecast.time_stamp_utc} is NOT a multiple of "
                          f"quantization {quantization}"
                      )

              print(f"{ts_name}: All forecasts are multiples of quantization {quantization}")
          else:
              print(f"{forecast_result.input.actuals.name}: No quantization detected")



In [10]:
assert_forecasts_match_quantization(results)

Demand: No quantization detected
Demand-Europe: No quantization detected
Demand-Europe-C011414575: No quantization detected
Demand-Europe-C024897037: All forecasts are multiples of quantization 2.0
Demand-Europe-C031595988: No quantization detected
Demand-Europe-C042056164: No quantization detected
Demand-Europe-C046568618: No quantization detected
Demand-Europe-C048529686: No quantization detected
Demand-North America: No quantization detected
Demand-North America-C010887784: All forecasts are multiples of quantization 3.0
Demand-North America-C011414575: No quantization detected
Demand-North America-C017968454: No quantization detected
Demand-North America-C024897037: No quantization detected
Demand-North America-C026568362: No quantization detected
Demand-North America-C031595988: No quantization detected
Demand-North America-C042056164: No quantization detected
Demand-North America-C046568618: No quantization detected
Demand-North America-C048097345: No quantization detected
Demand